# Working with AWS S3 as Object Storage (Key–Value Model)

This notebook demonstrates:
1. Connecting to AWS S3
2. Reading a CSV from S3
3. Writing a new CSV back to S3
4. Explicitly showing S3 objects as **key–value pairs**

In [1]:
!pip install boto3 pandas s3fs

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

In [2]:
import os
import pandas as pd
import boto3

## Configure AWS Credentials
**For demo/learning only**. Use IAM roles or secrets managers in production.

In [3]:
os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"

## Define Bucket and Object Keys
S3 uses **object keys**, not directories.

In [4]:
BUCKET_NAME = "gwu-data-engineering-example-bucket"
INPUT_KEY = "cancer_dataset_uae.csv"
OUTPUT_KEY = "cancer_dataset_uae_processed.csv"

## Read CSV from S3 into pandas

In [5]:
s3_input_path = f"s3://{BUCKET_NAME}/{INPUT_KEY}"
df = pd.read_csv(s3_input_path)
df.head()

/usr/local/lib/python3.12/dist-packages/fsspec/registry.py:286: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,Patient_ID,Age,Gender,Nationality,Emirate,Diagnosis_Date,Cancer_Type,Cancer_Stage,Treatment_Type,Treatment_Start_Date,Hospital,Primary_Physician,Outcome,Death_Date,Cause_of_Death,Smoking_Status,Comorbidities,Ethnicity,Weight,Height
0,PAT000001,69,Female,Emirati,Umm Al Quwain,2020-11-30,Liver,II,Radiation,2020-12-04,Sheikh Khalifa Hospital,Dr. VO41,Recovered,NaN,NaN,Non-Smoker,NaN,European,61,157
1,PAT000002,32,Male,Emirati,Umm Al Quwain,2015-10-10,Leukemia,III,Surgery,2015-11-05,Dubai Hospital,Dr. SM31,Recovered,NaN,NaN,Smoker,NaN,South Asian,80,175
2,PAT000003,89,Male,Emirati,Abu Dhabi,2018-02-13,Liver,III,Radiation,2018-08-03,Zayed Military Hospital,Dr. BC7,Under Treatment,NaN,NaN,Non-Smoker,NaN,South Asian,50,175
3,PAT000004,78,Female,Emirati,Abu Dhabi,2022-02-04,Lung,III,Radiation,2022-03-13,Cleveland Clinic Abu Dhabi,Dr. TC14,Recovered,NaN,NaN,Former Smoker,NaN,African,44,155
4,PAT000005,38,Female,Emirati,Fujairah,2019-12-03,Pancreatic,II,Chemotherapy,2020-02-29,Sheikh Khalifa Hospital,Dr. YS37,Recovered,NaN,NaN,Former Smoker,NaN,East Asian,101,160


## Transform the Data

In [6]:
roman_to_int = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4
}

df["Cancer_Stage_Int"] = df["Cancer_Stage"].map(roman_to_int)

## Write New CSV Back to S3
This creates a **new object** in S3.

In [7]:
s3_output_path = f"s3://{BUCKET_NAME}/{OUTPUT_KEY}"
df.to_csv(s3_output_path, index=False)
print("Wrote new object to:", s3_output_path)

Wrote new object to: s3://gwu-data-engineering-example-bucket/cancer_dataset_uae_processed.csv


## Demonstrate S3 as a Key–Value Store
Below we list objects and print their **key → metadata (value)**

In [8]:
s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=BUCKET_NAME)

for obj in response.get("Contents", []):
    print({
        "Key": obj["Key"],
        "Size(Bytes)": obj["Size"],
        "LastModified": obj["LastModified"].isoformat()
    })

{'Key': 'cancer_dataset_uae.csv', 'Size(Bytes)': 1727920, 'LastModified': '2026-02-07T00:46:50+00:00'}
{'Key': 'cancer_dataset_uae_processed.csv', 'Size(Bytes)': 1677701, 'LastModified': '2026-02-07T00:50:03+00:00'}


In [9]:
s3_output_path = f"s3://{BUCKET_NAME}/{OUTPUT_KEY}"
df.to_csv(s3_output_path, index=False)
print("Wrote new object to:", s3_output_path)

Wrote new object to: s3://gwu-data-engineering-example-bucket/cancer_dataset_uae_processed.csv


## Key Takeaway
- **Bucket** = namespace
- **Key** = unique identifier
- **Value** = binary object + metadata
- Objects are **immutable** (writes create new objects)